# Data Collection & Market Data Pipeline

Developed an asynchronous cryptocurrency market data collection pipeline using the KuCoin API.

Implemented concurrent API requests with Python asyncio and aiohttp to efficiently retrieve large volumes of historical market data.

Collected more than 365 days of daily OHLCV data for all actively traded cryptocurrency pairs.

Applied data validation, preprocessing, and formatting using Pandas.
Stored cleaned datasets in compressed Parquet format for efficient downstream quantitative research and backtesting.
Implemented retry logic, rate-limit handling, and automated data processing workflows.

# Сбор и построение пайплайна рыночных данных

Разработан асинхронный пайплайн для сбора криптовалютных рыночных данных с биржи KuCoin с использованием Python.

Реализованы параллельные API-запросы с использованием asyncio и aiohttp для эффективной загрузки больших объёмов данных.

Собраны исторические дневные OHLCV-данные (более 365 дней) по всем доступным торговым криптовалютным парам.

Выполнена очистка, валидация и предобработка данных с использованием Pandas.
Реализована система повторных запросов, обработка rate limit и отказоустойчивость при работе с API.

Данные сохранены в формате Parquet (сжатие gzip) для последующего количественного анализа и бэктестинга.

In [ ]:
# kucoin_collect.py — 100% WORKING | 100% РАБОЧИЙ
import asyncio, aiohttp, pandas as pd
from datetime import datetime, timedelta, UTC
from pathlib import Path
from tqdm.asyncio import tqdm_asyncio
import nest_asyncio

nest_asyncio.apply()

OUTPUT_DIR = Path("kucoin_daily_365d")
OUTPUT_DIR.mkdir(exist_ok=True)

API = "https://api.kucoin.com"


async def fetch(session, url, params=None):
    for _ in range(3):
        try:
            async with session.get(url, params=params, timeout=20) as r:
                if r.status == 429:
                    await asyncio.sleep(5)
                    continue
                if r.status == 200:
                    j = await r.json()
                    if j.get("code") == "200000":
                        return j.get("data", [])
        except:
            await asyncio.sleep(1)
    return []


async def get_symbols(session):
    data = await fetch(session, f"{API}/api/v1/symbols")
    return [s["symbol"] for s in data if s.get("enableTrading")]


async def get_daily_candles(symbol, session, sem):
    async with sem:
        end_ts = int(datetime.now(UTC).timestamp())
        start_ts = end_ts - 370 * 86400

        params = {
            "symbol": symbol,
            "startAt": start_ts,
            "endAt": end_ts,
            "type": "1day"
        }

        raw = await fetch(session, f"{API}/api/v1/market/candles", params)

        if not raw or len(raw) < 10:
            return

        df = pd.DataFrame(
            raw,
            columns=["ts", "open", "close", "high", "low", "volume", "turnover"]
        )

        df = df[["ts", "open", "close", "high", "low", "volume"]]
        df["ts"] = pd.to_datetime(df["ts"].astype(int), unit='s', utc=True)
        df.set_index("ts", inplace=True)
        df = df.astype(float)
        df = df.sort_index()

        path = OUTPUT_DIR / f"{symbol}.parquet"
        df.to_parquet(path, compression="gzip")


async def main():
    print("Starting data download from KuCoin | Начинаем загрузку данных с KuCoin")

    async with aiohttp.ClientSession() as session:
        symbols = await get_symbols(session)

        print(f"Active trading pairs found: {len(symbols)} | Найдено активных торговых пар: {len(symbols)}")

        sem = asyncio.Semaphore(30)
        tasks = [get_daily_candles(sym, session, sem) for sym in symbols]

        await tqdm_asyncio.gather(
            *tasks,
            desc="Completed | Готово",
            colour="green"
        )

        print(f"Data saved successfully | Данные успешно сохранены: {OUTPUT_DIR.resolve()}")


# ENTRY POINT
if __name__ == "__main__":
    asyncio.run(main())

Starting data download from KuCoin | Начинаем загрузку данных с KuCoin
Active trading pairs found: 1063 | Найдено активных торговых пар: 1063


Completed | Готово: 100%|██████████| 1063/1063 [00:36<00:00, 28.93it/s] 

Data saved successfully | Данные успешно сохранены: /content/kucoin_daily_365d


# DeFi Liquidity Dataset Collection & Preprocessing (DefiLlama)

Collected and processed a full-scale decentralized finance (DeFi) liquidity dataset from DefiLlama (~18,000 liquidity pools).

Retrieved the complete pool dataset via the DefiLlama yields API endpoint and converted raw JSON data into a structured Pandas DataFrame.

Processed a large-scale dataset of liquidity pools including key metrics such as APY, TVL, reward tokens, blockchain networks, and protocol information.
Performed feature engineering to enhance dataset usability for quantitative research:

Extracted and standardized reward token lists for easier analysis.
Generated direct URLs to individual liquidity pools for tracking and validation.
Created protocol-level links for systematic cross-protocol analysis.

Calculated base APY excluding reward incentives (apyBase) to separate organic yield from token emissions.

Optimized storage for large-scale analytics by saving the dataset in compressed Parquet format (gzip).

Prepared a structured dataset suitable for DeFi yield analysis, strategy filtering, and quantitative investment research.

# Сбор и подготовка полного датасета DeFi ликвидности (DefiLlama)

Загружен и обработан полный набор данных по DeFi ликвидности с платформы DefiLlama (~18,000 пулов).

Выполнена загрузка полного JSON-датасета через API DefiLlama (yields endpoint) с последующей конвертацией в Pandas DataFrame.

Обработан полный массив пулов ликвидности с сохранением всех доступных параметров (APY, TVL, награды, сеть, протоколы).

Проведена feature engineering: добавлены дополнительные аналитические признаки для дальнейшего анализа.

Формирование списка reward-токенов в читаемом виде.

Генерация прямых ссылок на пул и протокол для удобной навигации и анализа.
Расчёт базового APY без учёта наград (apyBase) для разделения органической доходности и стимулирующей эмиссии.

Выполнена оптимизация хранения данных и сохранение в формате Parquet с компрессией gzip для эффективной работы с большим объёмом данных.
Подготовлен структурированный датасет для дальнейшего количественного анализа DeFi доходностей, фильтрации стратегий и построения инвестиционных моделей.

In [ ]:
# STEP 1 of 2 | ШАГ 1 из 2: Download full DefiLlama pools dataset (Dec 2025)
import requests
import pandas as pd
from pathlib import Path

print("Downloading full pool dataset from DefiLlama (~18k rows) | Скачиваем полный список пулов DefiLlama (~18 тыс. строк)...")

url = "https://yields.llama.fi/pools"
response = requests.get(url, timeout=90)
response.raise_for_status()
raw_data = response.json()["data"]

pools = pd.DataFrame(raw_data)

print(f"Pools downloaded: {len(pools):,} | Скачано пулов: {len(pools):,}")

# === Feature engineering | Дополнительные признаки ===
def reward_names(tokens):
    if isinstance(tokens, list) and tokens:
        return ", ".join([str(t) for t in tokens if t])
    return ""

pools["rewardTokensNames"] = pools["rewardTokens"].apply(reward_names)

# Pool URL | Ссылка на пул
pools["poolLink"] = "https://defillama.com/yields/pool/" + pools["pool"]

# Protocol URL | Ссылка на протокол
pools["protocolLink"] = pools["project"].apply(
    lambda x: f"https://defillama.com/protocol/{str(x).replace(' ', '-').lower()}" if pd.notna(x) else ""
)

# Base APY (without rewards) | Базовый APY без наград
pools["apyBase"] = pools["apy"] - pools.get("apyReward", 0)

# Save dataset | Сохранение датасета
output_file = Path("defillama_all_pools_full.parquet")
pools.to_parquet(output_file, compression="gzip")

print(f"\nDONE | ГОТОВО!")
print(f"File saved | Файл сохранён: {output_file}")
print(f"Size | Размер: {output_file.stat().st_size / 1024**2:.1f} MB")
print(f"Rows | Строки: {len(pools):,}")
print(f"Columns | Колонки: {len(pools.columns)}")

# Preview | Пример данных
pools[["chain", "project", "symbol", "tvlUsd", "apy", "rewardTokensNames", "poolLink"]].head(10)

Pools downloaded: 16,224 | Скачано пулов: 16,224

DONE | ГОТОВО!
File saved | Файл сохранён: defillama_all_pools_full.parquet
Size | Размер: 2.1 MB
Rows | Строки: 16,224
Columns | Колонки: 31


,chain,project,symbol,tvlUsd,apy,rewardTokensNames,poolLink
0,Ethereum,lido,STETH,14968692373,2.35000,,https://defillama.com/yields/pool/747c1d2a-c66...
1,Ethereum,sky-lending,SUSDS,6159657221,3.60000,,https://defillama.com/yields/pool/d8c4eff5-c8a...
2,Ethereum,binance-staked-eth,WBETH,5868629197,2.40827,,https://defillama.com/yields/pool/80b8bf92-b95...
3,Ethereum,maple,USDC,3146026734,4.76901,,https://defillama.com/yields/pool/43641cf5-a92...
4,BSC,circle-usyc,USYC,2924922364,3.02889,,https://defillama.com/yields/pool/7c0a89c7-70c...
5,Ethereum,ether.fi-stake,WEETH,2871312006,2.90567,0x8F08B70456eb22f6109F57b8fafE862ED28E6040,https://defillama.com/yields/pool/46bd2bdf-6d9...
6,Ethereum,rocket-pool,RETH,2275274229,2.06391,,https://defillama.com/yields/pool/d4b3c522-612...
7,Ethereum,sparklend,WSTETH,2197286824,0.00000,,https://defillama.com/yields/pool/3b45941c-16c...
8,Base,morpho-blue,CBBTC,2142870821,0.00000,,https://defillama.com/yields/pool/7d33d57d-36d...
9,Ethereum,aave-v3,WBTC,1960686499,0.00541,,https://defillama.com/yields/pool/7e382157-b1b...


# Comprehensive DeFi Yield Analytics & Market Structure Analysis

Developed a full-scale quantitative analysis module for decentralized finance (DeFi) yield markets using DefiLlama liquidity pool data (~18,000–22,000 pools).

Built a multi-dimensional analytical engine to explore DeFi market structure across chains, protocols, and liquidity pools.

Performed cross-sectional analysis of key market metrics including TVL distribution, APY dynamics, reward structures, and risk exposure.

Identified and ranked top blockchain networks by number of liquidity pools and total TVL, revealing capital concentration across DeFi ecosystems.

Conducted token-level frequency analysis for both trading pairs and reward incentives to identify dominant assets in DeFi yield strategies.

Designed a systematic screening framework for high-yield liquidity pools with constraints on TVL, APY, and outlier filtering.

Analyzed protocol dominance and capital allocation across major DeFi ecosystems with structured ranking and link-based validation.

Decomposed yield into base APY and reward-based APY to distinguish organic yield from incentive-driven emissions.

Evaluated risk characteristics including stablecoin vs volatile pool segmentation and impermanent loss exposure distribution.

Performed temporal analysis of yield dynamics, including APY changes and momentum-based yield shifts where available.

Structured results into a reproducible research pipeline for DeFi market intelligence and quantitative strategy development.

# Комплексный анализ доходности DeFi и структуры рынка ликвидности

Разработан полноценный модуль количественного анализа рынка DeFi на основе данных DefiLlama (~18,000–22,000 пулов ликвидности).

Построен многомерный аналитический движок для исследования структуры DeFi-рынка по сетям, протоколам и пулам ликвидности.

Выполнен кросс-секционный анализ ключевых метрик: распределение TVL, динамика APY, структура наград и риск-профили.

Определены и ранжированы блокчейн-сети по количеству пулов и объёму капитала, выявлена концентрация ликвидности в экосистемах DeFi.

Проведен анализ частоты токенов в торговых парах и reward-инсентивов для выявления доминирующих активов в доходных стратегиях.

Реализован фильтр высокодоходных пулов с ограничениями по TVL, APY и исключением статистических выбросов.

Проведен анализ доминирования протоколов и распределения капитала между ключевыми DeFi-платформами.

Выполнено разделение доходности на базовый APY и наградной компонент для оценки органической доходности и эмиссионных стимулов.

Проанализированы рисковые характеристики, включая разделение пулов на стейблкоин и волатильные, а также экспозицию к impermanent loss.

Исследована динамика изменения доходности (APY momentum) при наличии соответствующих данных.

Результаты структурированы в воспроизводимый аналитический пайплайн для DeFi research и количественных стратегий.

In [ ]:
# FULL DEFI LLAMA ANALYSIS NOTEBOOK CELL (RU | EN VERSION)
import pandas as pd
from collections import Counter

df = pd.read_parquet("defillama_all_pools_full.parquet")

print("DEFILLAMA | DEFI LLAMA ANALYSIS ~18–22K POOLS | ПОЛНЫЙ АНАЛИЗ ПУЛОВ")
print("=" * 100)

# 1. CHAINS
print("\n1. CHAINS BY NUMBER OF POOLS | СЕТИ ПО КОЛИЧЕСТВУ ПУЛОВ:")
chain_count = df['chain'].value_counts()
chain_pct = (chain_count / len(df) * 100).round(2)

top_chains = pd.concat([chain_count, chain_pct], axis=1)
top_chains.columns = ["pools", "% of total | % от всех"]

print(top_chains.head(15).to_string())

# 2. TVL BY CHAIN
print("\n\n2. CHAINS BY TVL | СЕТИ ПО TVL (Total Value Locked):")
tvl_by_chain = df.groupby("chain")["tvlUsd"].sum().sort_values(ascending=False)
total_tvl = tvl_by_chain.sum()

tvl_by_chain_b = (tvl_by_chain / 1e9).round(2)
tvl_pct = (tvl_by_chain / total_tvl * 100).round(2)

top_tvl = pd.concat([tvl_by_chain_b, tvl_pct], axis=1)
top_tvl.columns = ["TVL (B$)", "% of DeFi | % от DeFi"]

print(top_tvl.head(15).to_string())

# 3. TOKENS FREQUENCY
print("\n\n3. TOP TOKENS IN POOLS | ТОП ТОКЕНОВ В ПУЛАХ:")
all_tokens = []

for sym in df['symbol'].dropna():
    tokens = [t.strip() for t in str(sym).split('-') if t.strip() and len(t.strip()) <= 12]
    all_tokens.extend(tokens)

token_stats = Counter(all_tokens).most_common(25)

for token, count in token_stats:
    print(f"{token:10} | {count:,} times | раз")

# 4. REWARD TOKENS
print("\n\n4. REWARD TOKENS | ТОКЕНЫ НАГРАД:")
reward_tokens = []

for rewards in df['rewardTokensNames'].dropna():
    if rewards:
        reward_tokens.extend([t.strip() for t in rewards.split(',') if t.strip()])

reward_stats = Counter(reward_tokens).most_common(15)

for token, count in reward_stats:
    print(f"{token:12} | {count:,} pools | пулов")

# 5. BEST FARMS
print("\n\n5. BEST YIELD POOLS | ЛУЧШИЕ YIELD ПУЛЫ (TVL ≥ $10M):")

best_farms = df[
    (df['tvlUsd'] >= 10_000_000) &
    (df['apy'] > 5) &
    (df['outlier'] == False)
].sort_values("apy", ascending=False).head(15)

for _, row in best_farms.iterrows():
    print(
        f"{row['apy']:6.1f}% APY | "
        f"{row['symbol'][:35]:35} | "
        f"{row['chain']:12} | "
        f"${row['tvlUsd']/1e6:6.1f}M TVL | "
        f"{row['project']:18} | "
        f"{row['poolLink']}"
    )

# 6. PROTOCOLS
print("\n\n6. TOP PROTOCOLS BY TVL | ТОП ПРОТОКОЛОВ ПО TVL:")

tvl_by_project = df.groupby("project")["tvlUsd"].sum().sort_values(ascending=False)
tvl_by_project_b = (tvl_by_project / 1e9).round(2)
project_pct = (tvl_by_project / total_tvl * 100).round(2)

top_projects = pd.concat([tvl_by_project_b, project_pct], axis=1)
top_projects.columns = ["TVL (B$)", "% of DeFi | % DeFi"]

top_projects["link"] = top_projects.index.map(
    lambda x: f"https://defillama.com/protocol/{str(x).replace(' ', '-').lower()}"
)

for idx, row in top_projects.head(15).iterrows():
    print(f"{row['TVL (B$)']:6.2f}B$ | {row['% of DeFi | % DeFi']:5.2f}% | {idx:30} | {row['link']}")

# 7. BASE vs REWARDS
print("\n\n7. BASE APY vs REWARDS | БАЗОВЫЙ APY И НАГРАДЫ:")

has_base = df['apyBase'].notna() & (df['apyBase'] > 0)
has_reward = df['apyReward'].notna() & (df['apyReward'] > 0)

print(f"Only Base APY | Только базовый APY: {has_base.sum():,} ({has_base.mean()*100:.1f}%)")
print(f"Only Rewards | Только награды: {(has_reward & ~has_base).sum():,}")
print(f"Both | Оба: {(has_base & has_reward).sum():,}")

print(f"Avg Base APY | Средний базовый APY: {df['apyBase'].mean():.2f}%")
print(f"Avg Rewards APY | Средний rewards APY: {df['apyReward'][df['apyReward']>0].mean():.2f}%")

# 8. STABLE vs VOLATILE
print("\n\n8. STABLE vs VOLATILE POOLS | СТЕЙБЛЫ И ВОЛАТИЛЬНЫЕ:")

stable = df[df['stablecoin'] == True]
volatile = df[df['stablecoin'] == False]

print(
    f"Stable pools | Стейбл пулы: {len(stable):,} ({len(stable)/len(df)*100:.1f}%) | "
    f"Avg APY: {stable['apy'].mean():.2f}% | TVL: ${stable['tvlUsd'].sum()/1e9:.2f}B"
)

print(
    f"Volatile pools | Волатильные: {len(volatile):,} ({len(volatile)/len(df)*100:.1f}%) | "
    f"Avg APY: {volatile['apy'].mean():.2f}% | TVL: ${volatile['tvlUsd'].sum()/1e9:.2f}B"
)

# 9. IL RISK
print("\n\n9. IMPERMANENT LOSS RISK | РИСК IL:")

print("Exposure distribution | Распределение экспозиции:")
print(df['exposure'].value_counts().to_string())

il_risk = (df['ilRisk'] == 'yes').sum()
print(f"\nIL risk pools | Пулы с IL риском: {il_risk:,} ({il_risk/len(df)*100:.1f}%)")

if 'il7d' in df.columns:
    il_pos = df[df['il7d'] > 0]['il7d']
    if len(il_pos) > 0:
        print(f"Avg positive IL 7d | Средний IL 7 дней: {il_pos.mean():.3f}%")

# 10. APY GROWTH
print("\n\n10. APY GROWTH (7D) | РОСТ APY ЗА 7 ДНЕЙ:")

if 'apyPct7D' in df.columns:
    growing = df[
        (df['apyPct7D'].notna()) &
        (df['tvlUsd'] > 5_000_000) &
        (df['outlier'] == False)
    ].sort_values("apyPct7D", ascending=False).head(10)

    for _, row in growing.iterrows():
        print(
            f"+{row['apyPct7D']:.1f}% | {row['apy']:.1f}% APY | "
            f"{row['symbol'][:30]:30} | {row['chain']:12} | {row['project']}"
        )

# 11. POOL CATEGORIES
print("\n\n11. POOL CATEGORIES | КАТЕГОРИИ ПУЛОВ:")

if 'poolMeta' in df.columns:
    meta_count = df['poolMeta'].value_counts().head(12)
    meta_tvl = (df.groupby('poolMeta')['tvlUsd'].sum().sort_values(ascending=False).head(12) / 1e9)

    for k in meta_count.index:
        print(f"{k} | pools: {meta_count[k]} | TVL: {meta_tvl.get(k, 0):.2f}B")

# 12. FINAL SUMMARY
print("\n" + "=" * 100)
print("DONE | ГОТОВО: FULL DEFI YIELD MAP 2025")
print(f"Total pools | Всего пулов: {len(df):,}")
print(f"Total TVL | Общий TVL: ${total_tvl/1e9:.2f}B")

DEFILLAMA | DEFI LLAMA ANALYSIS ~18–22K POOLS | ПОЛНЫЙ АНАЛИЗ ПУЛОВ

1. CHAINS BY NUMBER OF POOLS | СЕТИ ПО КОЛИЧЕСТВУ ПУЛОВ:
                pools  % of total | % от всех
chain                                        
Ethereum         5043                   31.08
Solana           2860                   17.63
Base             2568                   15.83
Arbitrum          967                    5.96
Polygon           672                    4.14
BSC               570                    3.51
Hyperliquid L1    456                    2.81
OP Mainnet        416                    2.56
Avalanche         319                    1.97
Sui               250                    1.54
Monad             198                    1.22
TON               188                    1.16
Starknet          154                    0.95
Sonic             118                    0.73
Osmosis           109                    0.67


2. CHAINS BY TVL | СЕТИ ПО TVL (Total Value Locked):
                TVL (B$)  % of DeFi |

# Interactive Volatility Forecasting & Risk Probability Engine

Developed an interactive quantitative analytics system for volatility forecasting and probabilistic risk estimation across cryptocurrency assets.

Built a multi-asset volatility analysis framework using historical daily price data stored in Parquet format.

Computed daily return distributions and percent-change dynamics for multiple cryptocurrency tokens.

Designed an interactive Jupyter-based UI using ipywidgets for real-time parameter tuning and analysis.

Implemented probabilistic modeling of price boundary breakouts based on historical return distributions.

Estimated likelihood of price movements exceeding configurable percentage channels using empirical distribution analysis.

Developed horizon-based risk modeling to simulate probability of price deviations across multiple time intervals.

Integrated dynamic token selection and manual input functionality for flexible research workflows.

Automated batch processing pipeline to compute volatility and risk metrics across all available assets.

Built a system to export full probabilistic forecasts into structured pickle files for downstream research and strategy development.

Enabled portfolio-wide volatility comparison and systematic risk evaluation across multiple crypto assets.

# Интерактивная система прогнозирования волатильности и оценки вероятностей риска

Разработана интерактивная система количественного анализа волатильности и вероятностной оценки рисков для криптовалютных активов.

Построен многоактивный фреймворк анализа волатильности на основе исторических дневных данных (Parquet).

Рассчитаны распределения дневных доходностей и процентных изменений для множества криптовалютных токенов.

Создан интерактивный интерфейс на базе ipywidgets для динамического изменения параметров анализа.

Реализовано вероятностное моделирование пробоя ценовых каналов на основе эмпирических распределений доходностей.

Оценена вероятность выхода цены за заданные процентные границы с использованием исторических данных.

Разработана модель горизонтов риска для оценки вероятности отклонений цены на разных временных интервалах.

Реализована система гибкого выбора токенов и ручного ввода параметров для исследовательского анализа.

Создан автоматизированный пайплайн расчёта волатильности и риск-метрик по всем доступным активам.

Реализован экспорт результатов в формате pickle для дальнейшего количественного анализа и стратегий.

Система позволяет проводить сравнительный анализ волатильности и риска между различными криптоактивами.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
import pickle
from datetime import datetime

# =========================
# DATA | ДАННЫЕ
# =========================
DATA_DIR = Path("kucoin_daily_365d")

# Collect tokens and daily % returns | Сбор токенов и дневных % изменений
ranges_list = []
tickers = []

for f in sorted(DATA_DIR.glob("*.parquet")):
    token = f.stem.replace("-USDT", "")
    tickers.append(token)
    df = pd.read_parquet(f)

    pct_change = df['close'].pct_change() * 100
    ranges_list.append(pct_change)

ranges = pd.concat(ranges_list, axis=1, keys=tickers)

# =========================
# WIDGETS | ВИДЖЕТЫ
# =========================

token_dropdown = widgets.Dropdown(
    options=sorted(ranges.columns),
    description='Token | Токен:',
    layout=widgets.Layout(width='300px')
)

token_text = widgets.Text(
    value='',
    description='Manual | Ручной ввод:',
    placeholder='e.g. BTC',
    layout=widgets.Layout(width='300px')
)

percent_widget = widgets.FloatText(
    value=5.0,
    description='Percent | Процент:',
    step=0.1
)

horizon_widget = widgets.IntRangeSlider(
    value=[1, 15],
    min=0,
    max=30,
    step=1,
    description='Horizon % | Горизонт %:',
    continuous_update=False
)

calc_button = widgets.Button(
    description='Calculate | Рассчитать',
    button_style='success'
)

pickle_button = widgets.Button(
    description='Save pickle | Сохранить pickle',
    button_style='warning'
)

output = widgets.Output()
pickle_output = widgets.Output()

# =========================
# HELPERS | ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
# =========================

def get_selected_token():
    t = token_text.value.strip().upper()
    if t and t in ranges.columns:
        return t
    return token_dropdown.value

def load_current_price(token):
    file_path = DATA_DIR / f"{token}-USDT.parquet"
    if not file_path.exists():
        return None
    df = pd.read_parquet(file_path)
    return float(df['close'].iloc[-1])

# =========================
# SINGLE TOKEN ANALYSIS | АНАЛИЗ ОДНОГО ТОКЕНА
# =========================

def calculate(_):
    with output:
        clear_output()

        token = get_selected_token()
        pct = float(percent_widget.value)
        lo, hi = horizon_widget.value

        if token not in ranges.columns:
            print("❌ Token not found | Токен не найден")
            return

        series = ranges[token].dropna()
        if len(series) < 200:
            print("❌ Not enough data | Недостаточно данных")
            return

        price = load_current_price(token)
        if price is None:
            print("❌ Price missing | Нет цены")
            return

        lower = price * (1 - pct / 100)
        upper = price * (1 + pct / 100)

        percentile = (series <= pct).mean() * 100
        probability = 100 - percentile

        print(f"Token | Токен: {token}")
        print(f"Price | Цена: {price:.6f}")
        print(f"Channel | Канал: ±{pct:.2f}%")
        print(f"  Lower | Нижняя граница: {lower:.6f}")
        print(f"  Upper | Верхняя граница: {upper:.6f}")
        print()
        print(f"Percentile | Перцентиль: {percentile:.2f}%")
        print(f"Break probability | Вероятность выхода: {probability:.2f}%")

        rows = []
        for p in range(lo, hi + 1):
            perc = (series <= p).mean() * 100
            prob = 100 - perc

            rows.append({
                "percent": p,
                "lower_price": price * (1 - p / 100),
                "upper_price": price * (1 + p / 100),
                "percentile": perc,
                "probability_break": prob
            })

        horizon_df = pd.DataFrame(rows)

        print("\nHorizon analysis | Анализ горизонта:")
        display(horizon_df.round(6))

# =========================
# PICKLE FOR ALL TOKENS | PICKLE ДЛЯ ВСЕХ ТОКЕНОВ
# =========================

def create_pickle(_):
    with pickle_output:
        clear_output()
        print("Creating full dataset pickle | Создание полного pickle...")

        pct = float(percent_widget.value)
        lo, hi = horizon_widget.value

        results = {}

        for token in ranges.columns:
            series = ranges[token].dropna()
            if len(series) < 200:
                continue

            price = load_current_price(token)
            if price is None:
                continue

            lower = price * (1 - pct / 100)
            upper = price * (1 + pct / 100)

            percentile = (series <= pct).mean() * 100
            probability = 100 - percentile

            horizon_rows = []
            for p in range(lo, hi + 1):
                perc = (series <= p).mean() * 100
                prob = 100 - perc

                horizon_rows.append({
                    "percent": p,
                    "lower_price": price * (1 - p / 100),
                    "upper_price": price * (1 + p / 100),
                    "percentile": perc,
                    "probability_break": prob
                })

            results[token] = {
                "current_price": float(price),
                "percent": pct,
                "lower_bound": float(lower),
                "upper_bound": float(upper),
                "percentile": float(percentile),
                "probability_break": float(probability),
                "horizon": pd.DataFrame(horizon_rows)
            }

        filename = f"volatility_forecast_{pct:.2f}pct_{datetime.now().strftime('%Y%m%d_%H%M%S')}.pkl"

        with open(filename, "wb") as f:
            pickle.dump(results, f)

        print(f"✅ Done | Готово")
        print(f"Tokens processed | Токенов: {len(results)}")
        print(f"File | Файл: {filename}")

# =========================
# BINDING | ПРИВЯЗКА
# =========================

calc_button.on_click(calculate)
pickle_button.on_click(create_pickle)

# =========================
# UI | ИНТЕРФЕЙС
# =========================

display(
    widgets.VBox([
        token_dropdown,
        token_text,
        percent_widget,
        horizon_widget,
        calc_button,
        pickle_button,
        output,
        pickle_output
    ])
)

# Interactive DeFi Yield Screening & Cross-Exchange Liquidity Analysis Engine

Developed an interactive quantitative screening system for decentralized finance (DeFi) yield opportunities with cross-exchange liquidity validation.

Built a dynamic filtering engine for large-scale DeFi datasets (~18,000+ liquidity pools) from DefiLlama.

Designed an interactive parameter-driven UI using ipywidgets for real-time filtering of yield opportunities.

Implemented multi-factor screening based on TVL, APY, blockchain network, reward structure, and asset composition.

Engineered token-level parsing logic to decompose liquidity pools into underlying asset pairs (tokenA / tokenB).

Integrated cross-exchange liquidity validation by matching DeFi assets with KuCoin-listed tokens.

Developed classification logic for stablecoin vs volatile asset pools to support risk-adjusted filtering strategies.

Enabled advanced query-based filtering by token pairs, chains, and reward conditions.

Built a system for exporting full filtered datasets into structured pickle format for further quantitative research.

Implemented automated identification of yield opportunities across multiple DeFi ecosystems with execution-ready data outputs.

Designed a research interface for systematic DeFi yield discovery and portfolio screening.

# Интерактивная система фильтрации DeFi доходностей и кросс-биржевого анализа ликвидности

Разработана интерактивная система количественного отбора DeFi-доходностей с проверкой ликвидности через несколько рынков.

Построен динамический фильтр по крупному датасету DeFi (~18,000+ пулов ликвидности DefiLlama).

Реализован интерактивный UI на базе ipywidgets для анализа и отбора доходных возможностей в реальном времени.

Внедрена многокритериальная фильтрация по TVL, APY, блокчейн-сетям, структуре наград и составу активов.

Реализована логика разбиения пулов на базовые токены (tokenA / tokenB) для детального анализа структуры ликвидности.

Интегрирована проверка ликвидности через сопоставление токенов с листингами KuCoin.

Разработана классификация пулов на стейблкоин и волатильные активы для риск-ориентированного отбора.

Добавлена расширенная фильтрация по токенам, сетям и структуре наград.
Реализован экспорт отфильтрованных данных в формат pickle для последующего количественного анализа.

Система позволяет автоматически выявлять доходные возможности в различных DeFi-экосистемах.

Создан исследовательский интерфейс для системного поиска и отбора инвестиционных стратегий в DeFi.

In [ ]:
# CELL 4 — FINAL INTERACTIVE FILTER DEFI LLAMA + KuCoin | ЯЧЕЙКА 4 — ФИНАЛЬНЫЙ ИНТЕРАКТИВНЫЙ ФИЛЬТР DEFI LLAMA + KuCoin

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
from pathlib import Path

# === LOAD DEFILLAMA DATA | ЗАГРУЗКА ДАННЫХ DEFILLAMA ===
print("Loading latest DefiLlama dataset... | Загружаем актуальный массив DefiLlama...")

df = pd.read_parquet("defillama_all_pools_full.parquet")

# Split symbol into tokenA and tokenB | Разделение symbol на tokenA и tokenB
if 'tokenA' not in df.columns or 'tokenB' not in df.columns:
    def split_tokens(symbol):
        parts = str(symbol).split('-')
        tokenA = parts[0].strip() if len(parts) > 0 else ""
        tokenB = parts[1].strip() if len(parts) > 1 else ""
        return pd.Series([tokenA, tokenB])

    df[['tokenA', 'tokenB']] = df['symbol'].apply(split_tokens)

# === CHECK KUCOIN LISTING | ПРОВЕРКА НАЛИЧИЯ НА KUCOIN ===
kucoin_dir = Path("kucoin_daily_365d")
kucoin_tickers = [f.stem.replace("-USDT", "").upper() for f in kucoin_dir.glob("*-USDT.parquet")]

df['tokenA_on_KuCoin'] = df['tokenA'].str.upper().apply(lambda x: x in kucoin_tickers)
df['tokenB_on_KuCoin'] = df['tokenB'].str.upper().apply(lambda x: x in kucoin_tickers)

# === STABLECOINS | СТЕЙБЛКОИНЫ ===
STABLECOINS = {
    "USDT", "USDC", "DAI", "FDUSD", "TUSD", "BUSD",
    "FRAX", "USDP", "LUSD", "USDJ", "GUSD", "SUSD"
}

# Reward flag | Флаг наличия наград
df['has_rewards'] = df['rewardTokensNames'].notna() & (df['rewardTokensNames'] != "")

print(f"Ready! Loaded {len(df):,} pools | Готово! Загружено {len(df):,} пулов\n")

# === WIDGETS | ВИДЖЕТЫ ===

tvl_slider = widgets.FloatLogSlider(
    value=1_000_000,
    base=10,
    min=3, max=10,
    step=0.1,
    description='Min TVL $ | Мин TVL $',
    continuous_update=False,
    readout=True,
    readout_format=',.0f'
)

apy_slider = widgets.FloatSlider(
    value=5,
    min=0,
    max=300,
    step=1,
    description='Min APY % | Мин APY %',
    continuous_update=False
)

chain_dropdown = widgets.Dropdown(
    options=['All'] + sorted(df['chain'].dropna().unique().tolist()),
    value='All',
    description='Chain | Сеть:'
)

reward_dropdown = widgets.Dropdown(
    options=['All', 'With Rewards', 'No Rewards'],
    value='All',
    description='Rewards | Награды:'
)

token1 = widgets.Text(
    value='',
    placeholder='e.g. USDT, BTC, ETH...',
    description='Token 1 | Токен 1:'
)

token2 = widgets.Text(
    value='',
    placeholder='e.g. USDC, WETH, SOL...',
    description='Token 2 | Токен 2:'
)

kucoin_only_checkbox = widgets.Checkbox(
    value=False,
    description='KuCoin only | Только KuCoin'
)

stable_only_checkbox = widgets.Checkbox(
    value=False,
    description='Stable-Stable only | Только стейбл–стейбл'
)

token_only_checkbox = widgets.Checkbox(
    value=False,
    description='Token-Token only | Только токен–токен'
)

# === SAVE ALL POOLS | СОХРАНИТЬ ВСЕ ПУЛЫ ===
save_all_pickle_button = widgets.Button(
    description='Save all pools pickle | Сохранить все пулы pickle',
    button_style='success',
    icon='download'
)

output_save_all = widgets.Output()

def on_save_all_pickle_clicked(b):
    with output_save_all:
        clear_output()

        cols = [
            "chain", "project", "symbol", "tvlUsd", "apy",
            "has_rewards", "poolLink",
            "tokenA", "tokenB",
            "tokenA_on_KuCoin", "tokenB_on_KuCoin"
        ]

        df[cols].to_pickle("all_defillama_pools_full.pickle")

        print("Saved successfully | Успешно сохранено: all_defillama_pools_full.pickle")
        print(f"Pools saved | Пулов сохранено: {len(df):,}")

save_all_pickle_button.on_click(on_save_all_pickle_clicked)

# === FILTER FUNCTION | ФУНКЦИЯ ФИЛЬТРАЦИИ ===

def filter_pools(tvl_min, apy_min, chain, rewards, tok1, tok2,
                 kucoin_only, stable_only, token_only):

    f = df.copy()

    if tvl_min > 1000:
        f = f[f['tvlUsd'] >= tvl_min]

    if apy_min > 0:
        f = f[f['apy'] >= apy_min]

    if chain != 'All':
        f = f[f['chain'] == chain]

    if rewards == 'With Rewards':
        f = f[f['has_rewards']]
    elif rewards == 'No Rewards':
        f = f[~f['has_rewards']]

    tok1 = tok1.strip().upper()
    tok2 = tok2.strip().upper()

    if tok1:
        f = f[
            f['symbol'].str.upper().str.contains(tok1) |
            f['tokenA'].str.upper().str.contains(tok1) |
            f['tokenB'].str.upper().str.contains(tok1)
        ]

    if tok2:
        f = f[
            f['symbol'].str.upper().str.contains(tok2) |
            f['tokenA'].str.upper().str.contains(tok2) |
            f['tokenB'].str.upper().str.contains(tok2)
        ]

    if kucoin_only:
        f = f[f['tokenA_on_KuCoin'] & f['tokenB_on_KuCoin']]

    if stable_only:
        f = f[
            f['tokenA'].str.upper().isin(STABLECOINS) &
            f['tokenB'].str.upper().isin(STABLECOINS)
        ]

    if token_only:
        f = f[
            ~f['tokenA'].str.upper().isin(STABLECOINS) &
            ~f['tokenB'].str.upper().isin(STABLECOINS)
        ]

    f = f.sort_values("apy", ascending=False)

    clear_output(wait=True)
    print(f"Pools found | Найдено пулов: {len(f):,} из {len(df):,}\n")

    if f.empty:
        print("No results found | Ничего не найдено")
        return

    cols = [
        "chain", "project", "symbol", "tvlUsd", "apy",
        "has_rewards", "poolLink",
        "tokenA", "tokenB",
        "tokenA_on_KuCoin", "tokenB_on_KuCoin"
    ]

    styled = f[cols].style.format({
        "tvlUsd": "${:,.0f}",
        "apy": "{:.2f}%",
        "poolLink": lambda x: f'<a href="{x}" target="_blank">Open | Открыть</a>'
    })

    display(styled)

# === RUN UI | ЗАПУСК ИНТЕРФЕЙСА ===

interactive_filter = widgets.interactive(
    filter_pools,
    tvl_min=tvl_slider,
    apy_min=apy_slider,
    chain=chain_dropdown,
    rewards=reward_dropdown,
    tok1=token1,
    tok2=token2,
    kucoin_only=kucoin_only_checkbox,
    stable_only=stable_only_checkbox,
    token_only=token_only_checkbox
)

display(save_all_pickle_button, output_save_all)
display(interactive_filter)

Loading latest DefiLlama dataset... | Загружаем актуальный массив DefiLlama...
Ready! Loaded 16,224 pools | Готово! Загружено 16,224 пулов



Button(button_style='success', description='Save all pools pickle | Сохранить все пулы pickle', icon='download…

Output()

interactive(children=(FloatLogSlider(value=1000000.0, continuous_update=False, description='Min TVL $ | Мин TV…

# Time-to-Target Volatility & Price Movement Dynamics Engine

Developed a quantitative analysis system for modeling time-to-target price movements and volatility dynamics across cryptocurrency markets.

Built an event-based volatility analysis framework using historical daily OHLC data from multiple cryptocurrency assets.

Designed a probabilistic model to estimate time required for price to reach predefined percentage thresholds (±X% moves).

Implemented a brute-force simulation of historical price paths to measure distribution of time-to-event outcomes.

Computed statistical measures including mean, median, standard deviation, and occurrence frequency of price target hits.

Developed interactive Jupyter-based controls for real-time analysis of volatility ranges and asset selection.

Extended the system to perform batch processing across all available cryptocurrency assets in the dataset.

Generated cross-asset volatility comparison dataset capturing time-to-move characteristics for portfolio-level analysis.

Exported structured results into pickle format for further quantitative research and strategy development.

Enabled systematic evaluation of market efficiency through empirical time-to-move distributions.

# Система анализа времени достижения ценовых целей и динамики волатильности

Разработана количественная система анализа времени достижения ценовых уровней и динамики волатильности криптовалютных активов.

Построен событийный фреймворк анализа волатильности на основе исторических дневных OHLC данных.

Реализована вероятностная модель оценки времени достижения заданных процентных движений цены (±X%).

Проведено моделирование исторических ценовых траекторий для оценки распределения времени до события.

Рассчитаны статистические показатели: среднее, медиана, стандартное отклонение и частота достижения целей.

Создан интерактивный интерфейс на базе Jupyter widgets для анализа параметров волатильности в реальном времени.

Реализован массовый расчёт по всем доступным криптоактивам для построения сравнительной базы волатильности.

Сформирован кросс-активный датасет характеристик времени движения цены для портфельного анализа.

Результаты экспортируются в формат pickle для последующего количественного моделирования и стратегий.

Система позволяет эмпирически оценивать эффективность рынка через распределение времени достижения ценовых целей.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

# DATA PATH | ПУТЬ К ДАННЫМ
DATA_DIR = Path("kucoin_daily_365d")

# WIDGETS | ВИДЖЕТЫ
symbol_text = widgets.Text(value='ETH-USDT', description='Symbol (e.g. ETH-USDT) | Символ:')
min_percent = widgets.IntSlider(value=5, min=1, max=20, step=1, description='Min % | Мин %:')
max_percent = widgets.IntSlider(value=15, min=5, max=50, step=1, description='Max % | Макс %:')
current_price_slider = widgets.FloatSlider(
    value=0, min=0, max=100000, step=0.01,
    description='Current price $ | Текущая цена $'
)

output = widgets.Output()

# BUTTONS | КНОПКИ
calculate_button = widgets.Button(description="Calculate | Расчёт", button_style='primary')
save_all_button = widgets.Button(
    description="Save pickle (all tokens) | Сохранить pickle (все токены)",
    button_style='success',
    icon='download'
)

save_output = widgets.Output()


# =========================
# CORE FUNCTION | ОСНОВНАЯ ФУНКЦИЯ
# =========================
def calculate_days_to_percent(df, percent):
    closes = df['close'].values
    days_list = []

    for i in range(len(closes) - 1):
        current = closes[i]
        target_up = current * (1 + percent / 100)
        target_down = current * (1 - percent / 100)

        days = 0
        for j in range(i + 1, len(closes)):
            days += 1
            if closes[j] >= target_up or closes[j] <= target_down:
                days_list.append(days)
                break

    return days_list


# =========================
# SINGLE TOKEN ANALYSIS | АНАЛИЗ ОДНОГО ТОКЕНА
# =========================
def update(_):
    with output:
        output.clear_output(wait=True)

        symbol = symbol_text.value.upper()
        min_p = min_percent.value
        max_p = max_percent.value
        current_price = current_price_slider.value

        file_path = DATA_DIR / f"{symbol}.parquet"

        if not file_path.exists():
            print(f"File not found | Файл не найден: {file_path}")
            return

        df = pd.read_parquet(file_path).sort_index()

        if len(df) < 10:
            print(f"Not enough data | Недостаточно данных: {symbol}")
            return

        last_price = df['close'].iloc[-1]

        if current_price == 0:
            current_price = last_price
            current_price_slider.value = last_price

        print(f"Symbol | Символ: {symbol}")
        print(f"Last price | Последняя цена: {last_price:.2f} $")
        print(f"Used price | Используемая цена: {current_price:.2f} $")
        print(f"Data range | Диапазон данных: {len(df)} days\n")

        results = []

        for percent in tqdm(range(min_p, max_p + 1), desc="Processing % | Расчёт %", disable=True):

            days_list = calculate_days_to_percent(df, percent)

            if not days_list:
                results.append([
                    percent, np.nan, np.nan, np.nan, 0,
                    current_price * (1 + percent / 100),
                    current_price * (1 - percent / 100)
                ])
                continue

            results.append([
                percent,
                np.median(days_list),
                np.mean(days_list),
                np.std(days_list),
                len(days_list),
                current_price * (1 + percent / 100),
                current_price * (1 - percent / 100)
            ])

        table_df = pd.DataFrame(results, columns=[
            'Percent | %',
            'Median days | Медиана дней',
            'Mean days | Среднее дней',
            'Std | Разброс',
            'Cases | Кол-во случаев',
            'Upper price | Верхняя граница',
            'Lower price | Нижняя граница'
        ]).set_index('Percent | %')

        display(Markdown("### Time to reach ±X% move | Время до достижения ±X% движения"))
        display(
            table_df.style.format({
                'Median days | Медиана дней': '{:.1f}',
                'Mean days | Среднее дней': '{:.1f}',
                'Std | Разброс': '{:.1f}',
                'Cases | Кол-во случаев': '{:,}',
                'Upper price | Верхняя граница': '{:.2f}',
                'Lower price | Нижняя граница': '{:.2f}'
            }).background_gradient(cmap='Blues', subset=[
                'Median days | Медиана дней',
                'Mean days | Среднее дней'
            ])
        )


calculate_button.on_click(update)


# =========================
# MASS ANALYSIS | МАССОВЫЙ АНАЛИЗ
# =========================
def save_all_tokens_analysis(_):
    with save_output:
        save_output.clear_output()

        print("Running full dataset analysis | Запуск анализа всех токенов...")
        print("This may take time | Это займёт некоторое время...\n")

        min_p = min_percent.value
        max_p = max_percent.value

        all_results = []

        parquet_files = list(DATA_DIR.glob("*-USDT.parquet"))

        if not parquet_files:
            print("No data files found | Нет файлов данных")
            return

        for file_path in tqdm(parquet_files, desc="Tokens | Токены", disable=True):

            symbol = file_path.stem

            try:
                df = pd.read_parquet(file_path).sort_index()

                if len(df) < 10:
                    continue

                current_price = df['close'].iloc[-1]

                for percent in range(min_p, max_p + 1):
                    days_list = calculate_days_to_percent(df, percent)

                    if not days_list:
                        median = mean = std = np.nan
                        count = 0
                    else:
                        median = np.median(days_list)
                        mean = np.mean(days_list)
                        std = np.std(days_list)
                        count = len(days_list)

                    all_results.append([
                        symbol,
                        percent,
                        median,
                        mean,
                        std,
                        count,
                        current_price * (1 + percent / 100),
                        current_price * (1 - percent / 100),
                        current_price
                    ])

            except Exception as e:
                print(f"Error | Ошибка: {symbol} -> {e}")

        if not all_results:
            print("No results | Нет результатов")
            return

        final_df = pd.DataFrame(all_results, columns=[
            'symbol',
            'percent',
            'median_days',
            'mean_days',
            'std',
            'count',
            'upper_price',
            'lower_price',
            'current_price'
        ])

        filename = "all_tokens_volatility_analysis.pickle"
        final_df.to_pickle(filename)

        print(f"Done | Готово: {filename}")
        print(f"Tokens | Токенов: {final_df['symbol'].nunique()}")
        print(f"Rows | Строк: {len(final_df):,}")
        print(f"Range | Диапазон: {min_p}-{max_p}%")


save_all_button.on_click(save_all_tokens_analysis)


# =========================
# DISPLAY | ОТОБРАЖЕНИЕ
# =========================
display(
    symbol_text,
    min_percent,
    max_percent,
    current_price_slider,
    calculate_button,
    output
)

display(widgets.HTML("<hr><h3>Mass analysis | Массовый анализ</h3>"))
display(save_all_button)
display(save_output)

Text(value='ETH-USDT', description='Symbol (e.g. ETH-USDT) | Символ:')

IntSlider(value=5, description='Min % | Мин %:', max=20, min=1)

IntSlider(value=15, description='Max % | Макс %:', max=50, min=5)

FloatSlider(value=0.0, description='Current price $ | Текущая цена $', max=100000.0, step=0.01)

Button(button_style='primary', description='Calculate | Расчёт', style=ButtonStyle())

Output()

HTML(value='<hr><h3>Mass analysis | Массовый анализ</h3>')

Button(button_style='success', description='Save pickle (all tokens) | Сохранить pickle (все токены)', icon='d…

Output()

# Proportional Portfolio Rebalancing Decision Engine

Developed an interactive portfolio rebalancing system for dynamic asset allocation based on proportional deviation between assets.

Built a rule-based portfolio allocation engine to compute optimal rebalancing between two assets based on target proportional weights.

Designed a mathematical framework to translate proportional imbalances into executable buy/sell actions.

Implemented dynamic capital allocation logic to calculate target portfolio distribution based on deviation from equilibrium.

Generated real-time rebalancing signals including required buy/sell amounts per asset.

Developed an interactive Jupyter-based interface for scenario testing and allocation optimization.

Enabled simulation of portfolio drift correction under varying asset weight distributions.

Designed system to support decision-making for systematic portfolio maintenance strategies.

Provided actionable outputs for execution-ready rebalancing operations in quantitative portfolios.

# Система пропорционального ребалансинга портфеля

Разработана интерактивная система ребалансинга портфеля на основе отклонений пропорций между активами.

Построен rule-based движок распределения капитала для двух активов с расчётом оптимального ребалансинга.

Разработана математическая модель преобразования отклонений пропорций в торговые действия (buy/sell).

Реализована логика расчёта целевого распределения капитала на основе текущего дисбаланса портфеля.

Формируются сигналы ребалансинга с точными объёмами покупки и продажи для каждого актива.

Создан интерактивный интерфейс на базе Jupyter widgets для тестирования сценариев распределения.

Смоделирована корректировка дрейфа портфеля при изменении весов активов.
Система поддерживает принятие решений для регулярного управления портфелем.
Выходные данные формируются в виде готовых инструкций для исполнения ребалансинга.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# ===============================
# REBALANCE FUNCTION | ФУНКЦИЯ РЕБАЛАНСА
# ===============================
def rebalance_by_proportion(prop_token1, prop_token2, current_token1, current_token2):
    total = current_token1 + current_token2

    if total == 0:
        return "Error | Ошибка: Total capital is 0 | Общий капитал = 0"

    # Proportion delta | Разница пропорций
    delta = (prop_token2 - prop_token1) / prop_token1

    # Base 50/50 | Базовое распределение 50/50
    base = total / 2

    # Adjustment | Коррекция
    adjustment = abs(total * delta) / 2

    if delta >= 0:
        target_t1 = base - adjustment
        target_t2 = base + adjustment
    else:
        target_t1 = base + adjustment
        target_t2 = base - adjustment

    sell_t1 = max(0, current_token1 - target_t1)
    buy_t1  = max(0, target_t1 - current_token1)

    sell_t2 = max(0, current_token2 - target_t2)
    buy_t2  = max(0, target_t2 - current_token2)

    return {
        "total": round(total, 2),
        "target_t1": round(target_t1, 2),
        "target_t2": round(target_t2, 2),
        "sell_t1": round(sell_t1, 2),
        "buy_t1": round(buy_t1, 2),
        "sell_t2": round(sell_t2, 2),
        "buy_t2": round(buy_t2, 2),
        "delta_pct": round(delta * 100, 2)
    }


# ===============================
# WIDGETS | ВИДЖЕТЫ
# ===============================
prop_t1 = widgets.FloatText(value=100, description='Prop Token1 | Пропорция 1')
prop_t2 = widgets.FloatText(value=110, description='Prop Token2 | Пропорция 2')

curr_t1 = widgets.FloatText(value=1000, description='Current Token1 $ | Текущий Token1 $')
curr_t2 = widgets.FloatText(value=100, description='Current Token2 $ | Текущий Token2 $')

calc_button = widgets.Button(
    description="Calculate | Рассчитать",
    button_style='success'
)

output = widgets.Output()


# ===============================
# CALLBACK | ОБРАБОТЧИК
# ===============================
def on_calculate_clicked(b):
    with output:
        clear_output()

        result = rebalance_by_proportion(
            prop_t1.value,
            prop_t2.value,
            curr_t1.value,
            curr_t2.value
        )

        if isinstance(result, str):
            print(result)
            return

        print("Portfolio Rebalance Result | РЕЗУЛЬТАТ РЕБАЛАНСА\n")

        print(f"Total capital | Общий капитал: {result['total']} $")
        print(f"Delta | Разница: {result['delta_pct']} %\n")

        print("Target allocations | Целевые значения:")
        print(f"Token1 → {result['target_t1']} $")
        print(f"Token2 → {result['target_t2']} $\n")

        print("Actions | Действия:")

        if result['sell_t1'] > 0:
            print(f"Sell Token1 | Продать Token1: {result['sell_t1']} $")
        if result['buy_t1'] > 0:
            print(f"Buy Token1 | Купить Token1: {result['buy_t1']} $")

        if result['sell_t2'] > 0:
            print(f"Sell Token2 | Продать Token2: {result['sell_t2']} $")
        if result['buy_t2'] > 0:
            print(f"Buy Token2 | Купить Token2: {result['buy_t2']} $")


calc_button.on_click(on_calculate_clicked)


# ===============================
# UI | ИНТЕРФЕЙС
# ===============================
display(
    widgets.VBox([
        widgets.HTML("<h3>⚖️ Rebalance by Proportion | Ребаланс по пропорциям</h3>"),
        prop_t1,
        prop_t2,
        curr_t1,
        curr_t2,
        calc_button,
        output
    ])
)

# Delta-Neutral Liquidity Provision & Hedging Strategy Simulator

Developed an interactive quantitative simulation framework for delta-neutral liquidity provision and hedging strategies in decentralized finance (DeFi).

Built a pricing and risk simulation model for liquidity pool positions under bounded price ranges.

Implemented a delta-neutral hedging framework combining liquidity provision with synthetic short exposure.

Modeled PnL decomposition of LP positions across upper and lower price bounds.
Simulated payoff structures of liquidity pools under dynamic price movements.
Derived optimal hedge sizing in tokens and USD terms to neutralize directional exposure.

Designed interactive parameter controls for real-time adjustment of price range, liquidity composition, and market conditions.

Visualized full PnL surface for LP position, hedge position, and combined portfolio outcome.

Implemented dynamic rebalance mechanism for adjusting liquidity ranges based on price movement.

Enabled scenario-based analysis of impermanent loss mitigation strategies.
Provided a research tool for studying automated market maker (AMM) behavior and hedged yield strategies.

# Симулятор дельта-нейтральной стратегии и хеджирования ликвидности

Разработана интерактивная система моделирования дельта-нейтральных стратегий предоставления ликвидности и хеджирования в DeFi.

Построена модель оценки PnL позиций ликвидности в заданном ценовом диапазоне.
Реализован механизм дельта-нейтрального хеджирования через сочетание LP-позиции и синтетического шорта.

Смоделирована структура прибыли и убытков (PnL) при движении цены внутри и за пределами диапазона.

Проведено разложение доходности LP-позиций на компоненты по верхней и нижней границе диапазона.

Рассчитаны параметры оптимального хеджа в токенах и USD для нейтрализации направленного риска.

Создан интерактивный интерфейс для динамического изменения параметров стратегии.
Визуализирована поверхность прибыли для LP, хеджа и итогового портфеля.

Реализован механизм ребалансировки диапазона при изменении рыночной цены.
Проведён анализ стратегий снижения impermanent loss.

Система служит инструментом исследования поведения AMM и хеджированных доходных стратегий.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np

# ==============================
# Simplified model: full range rebalance at boundaries
# Упрощённая модель: полный ребаланс на границах диапазона
# ==============================
def calculate_delta_neutral(P0, L, U, USDC_pool, Tokens_value):
    if L >= P0 or U <= P0 or L >= U:
        return None

    Token_pool = Tokens_value / P0 if P0 > 0 else 0
    C_total = USDC_pool + Tokens_value

    # At upper bound U: entire pool → USDC
    # На верхней границе U: весь пул → USDC
    value_at_U = USDC_pool + Token_pool * U
    Profit_pool_U = value_at_U - C_total

    # At lower bound L: entire pool → tokens
    # На нижней границе L: весь пул → токены
    value_at_L = USDC_pool + Token_pool * L
    Profit_pool_L = value_at_L - C_total

    # Short position sizing in tokens
    # Размер шорта в токенах
    Q_short_tokens = Profit_pool_U / (U - P0) if (U - P0) > 0 else 0
    Q_short_USD = Q_short_tokens * P0

    # Short profit at L
    # Прибыль шорта на L
    Profit_short_L = Q_short_tokens * (P0 - L)

    # PnL functions
    # Функции PnL
    def pnl_pool(P):
        if P <= L:
            return value_at_L - C_total
        elif P >= U:
            return value_at_U - C_total
        else:
            return (P - P0) * (USDC_pool / P0 - Token_pool)

    def pnl_short(P):
        return Q_short_tokens * (P0 - P)

    def pnl_total(P):
        return pnl_pool(P) + pnl_short(P)

    return {
        'C_total': C_total,
        'P0': P0,
        'L': L,
        'U': U,
        'USDC_pool': USDC_pool,
        'Tokens_value': Tokens_value,
        'Token_pool': Token_pool,
        'Q_short_USD': Q_short_USD,
        'Q_short_tokens': Q_short_tokens,
        'Profit_pool_U': Profit_pool_U,
        'Profit_pool_L': Profit_pool_L,
        'Profit_short_L': Profit_short_L,
        'pnl_pool': pnl_pool,
        'pnl_short': pnl_short,
        'pnl_total': pnl_total
    }

# ==============================
# Widgets
# Виджеты
# ==============================
P0_slider = widgets.FloatSlider(value=100, min=10, max=2000, step=5, description='Price $ | Цена $')
L_slider = widgets.FloatSlider(value=80, min=1, max=2000, step=5, description='Lower L | Нижняя L')
U_slider = widgets.FloatSlider(value=120, min=10, max=3000, step=5, description='Upper U | Верхняя U')
USDC_slider = widgets.FloatSlider(value=500, min=0, max=50000, step=10, description='USDC pool | USDC пул')
Tokens_slider = widgets.FloatSlider(value=200, min=0, max=50000, step=10, description='Tokens $ | Токены $')

rebalance_button = widgets.Button(description="Rebalance | Ребаланс", button_style='success')
output = widgets.Output()

# ==============================
# Update function
# Обновление расчёта
# ==============================
def update(change=None):
    with output:
        output.clear_output(wait=True)

        P0 = P0_slider.value
        L = L_slider.value
        U = U_slider.value
        USDC_pool = USDC_slider.value
        Tokens_value = Tokens_slider.value

        res = calculate_delta_neutral(P0, L, U, USDC_pool, Tokens_value)

        if res is None:
            print("Error | Ошибка: L < P0 < U required")
            return

        print("┌──────────────────────────────────────────────┐")
        print(f" Total capital | Общий капитал: {res['C_total']:.2f} $")
        print(f" Price | Цена: {res['P0']:.2f} $")
        print(f" USDC pool | USDC пул: {res['USDC_pool']:.2f} $")
        print(f" Tokens | Токены: {res['Tokens_value']:.2f} $ ({res['Token_pool']:.2f} units)")
        print(f" Range | Диапазон: {res['L']:.2f} – {res['U']:.2f}")
        print(f" Short size | Размер шорта: {res['Q_short_USD']:.2f} $")
        print("└──────────────────────────────────────────────┘")

        print(f"\nUpper bound U | Верхняя граница U:")
        print(f" Pool PnL | Пул PnL: +{res['Profit_pool_U']:.2f} $")
        print(f" Short PnL | Шорт PnL: -{res['Profit_pool_U']:.2f} $")
        print(" Net | Итог: 0 $")

        print(f"\nLower bound L | Нижняя граница L:")
        print(f" Pool PnL | Пул PnL: {res['Profit_pool_L']:.2f} $")
        print(f" Short PnL | Шорт PnL: +{res['Profit_short_L']:.2f} $")
        print(f" Total | Итог: {res['Profit_short_L'] + res['Profit_pool_L']:+.2f} $")

        print("\nIncome source | Источник дохода: fees | комиссии")

        P_values = np.linspace(L * 0.8, U * 1.2, 300)
        pnl_p = [res['pnl_pool'](p) for p in P_values]
        pnl_s = [res['pnl_short'](p) for p in P_values]
        pnl_t = [res['pnl_total'](p) for p in P_values]

        plt.figure(figsize=(12, 6))
        plt.plot(P_values, pnl_p, label='Pool PnL | Пул')
        plt.plot(P_values, pnl_s, label='Short PnL | Шорт')
        plt.plot(P_values, pnl_t, label='Total PnL | Итог', linestyle='--', linewidth=2)

        plt.axhline(0, color='black')
        plt.axvline(P0, linestyle='--', label='P0')
        plt.axvline(L, linestyle=':', label='L')
        plt.axvline(U, linestyle=':', label='U')

        plt.title('Delta-neutral strategy | Дельта-нейтральная стратегия')
        plt.xlabel('Price | Цена')
        plt.ylabel('PnL')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.show()

# ==============================
# Rebalance
# Ребаланс
# ==============================
def rebalance(b):
    current_P0 = P0_slider.value
    width = (U_slider.value - L_slider.value) / 2

    new_P0 = current_P0
    new_L = new_P0 * (1 - width / new_P0)
    new_U = new_P0 * (1 + width / new_P0)

    P0_slider.value = new_P0
    L_slider.value = new_L
    U_slider.value = new_U

    with output:
        print(f"Rebalanced | Ребаланс выполнен: {new_L:.2f} – {new_U:.2f}")

rebalance_button.on_click(rebalance)

for slider in [P0_slider, L_slider, U_slider, USDC_slider, Tokens_slider]:
    slider.observe(update, names='value')

display(P0_slider, L_slider, U_slider, USDC_slider, Tokens_slider, rebalance_button, output)

update(None)

FloatSlider(value=100.0, description='Price $ | Цена $', max=2000.0, min=10.0, step=5.0)

FloatSlider(value=80.0, description='Lower L | Нижняя L', max=2000.0, min=1.0, step=5.0)

FloatSlider(value=120.0, description='Upper U | Верхняя U', max=3000.0, min=10.0, step=5.0)

FloatSlider(value=500.0, description='USDC pool | USDC пул', max=50000.0, step=10.0)

FloatSlider(value=200.0, description='Tokens $ | Токены $', max=50000.0, step=10.0)

Button(button_style='success', description='Rebalance | Ребаланс', style=ButtonStyle())

Output()